# 01. Scopes and closures

**Tier:** Deep. **Why you'd use this:** A factory returns functions that should each remember their own `n`, but a loop accidentally shares one name. Closures and the LEGB rule explain which `n` you get.

**After this lesson you can:** say which `n` a function sees; capture a loop variable on purpose; explain why a class-body comprehension cannot see the class.
About 30 minutes. **Uses:** [Scope](../../course/05-functions/03-scope-lesson.ipynb), [Numbers and strings](../05-numbers/01-numbers-strings-lesson.ipynb)


## Quick review

1. Why is `0.1 + 0.2` not exactly `0.3` in binary floats?
2. Does `s.strip()` change `s` in place?

<details><summary>Answers</summary>

Those fractions are not exact in binary. No — strings are immutable; methods return new strings.
</details>


## The idea

A name is looked up in four places, in order: the local frame, the enclosing function frames, the module globals, then builtins. That order is LEGB. If any assignment to the name appears in a function, the name is local to that function everywhere in it, including lines before the assignment.

A nested function that uses a name from an enclosing function closes over that name. The closure holds a cell, a box for the variable, not a copy of the value at definition time. The read happens when the inner function runs. A loop variable is one cell, so every function created in the loop sees the value left in it at the end.

<details><summary>Explain more</summary>

`global` writes the module slot. `nonlocal` writes an enclosing function's cell. Neither creates a local. A class body is a namespace while the class statement runs, and it is not an enclosing function, so a comprehension inside the class cannot see the class's own names.

`function.__closure__` is the tuple of cells the function captured. `cell_contents` is the current value in that box.
</details>

<details><summary>If you already write Java, JavaScript, or C#</summary>

This is lexical scope with a mutable binding. A `var` in a JavaScript `for` loop has the same late read. A default argument is the usual way to snapshot the value, because defaults are evaluated when the function is defined.
</details>


## See it work

A tariff builder returns one function per rate. Each function must multiply by the rate from its own iteration, not by whatever rate is left when the customer is billed. A default argument is evaluated when `def` runs, so it snapshots the rate into a local of the inner function.


In [1]:
def tariffs(rates: list[int]) -> list:
    # 1. One function per rate, holding that rate as a default.
    billers = []
    for rate in rates:

        def bill(units: int, rate: int = rate) -> int:
            return units * rate

        billers.append(bill)
    return billers


billers = tariffs([1, 2, 3])
print([bill(10) for bill in billers])
print(billers[0].__defaults__)

[10, 20, 30]
(1,)


| iteration | default stored on `bill` | later `bill(10)` |
| --- | --- | --- |
| rate 1 | `(1,)` | 10 |
| rate 2 | `(2,)` | 20 |
| rate 3 | `(3,)` | 30 |

```
billers[0] defaults --> 1     not the loop cell
billers[2] defaults --> 3
```

The loop cell named `rate` still ends at 3. The defaults are separate locals, filled at each `def`.


## Check yourself

Answer these from the example above before you run the cell.


1. Why do the three reporters share one value instead of three?
2. What does `cell_contents` show that the source of `bump` does not?
3. Why can't the class-body comprehension see `code`?

<details><summary>Answers</summary>

1. There is one local named `number` and one cell for it. Defining a function does not copy the current integer into the function. The call reads the cell later.
2. The source names `count`. The cell holds the integer that name currently points at, including after `nonlocal` updates it. The function object keeps that box alive after `counter` has returned.
3. The comprehension is a nested function. Class bodies are not enclosing function scopes, so LEGB never consults them. `code` exists as a class attribute only after the class statement finishes, and the comprehension does not look there.
</details>


In [2]:
def make_reporters():
    reporters = []
    for number in range(3):

        def report():
            return number

        reporters.append(report)
    return reporters


print([report() for report in make_reporters()])

[2, 2, 2]


<details><summary>Check your answers</summary>

- The list is `[2, 2, 2]`, not `[0, 1, 2]`. Each `report` reads the one cell named `number`. The loop finishes with `2` in that cell. The calls happen after the loop, so every function sees `2`.
</details>


## Why it works this way

Lookup is a walk of namespaces. The local namespace is the frame's array of slots — the first place and the cheap place. An enclosing function's names live in cells on the heap, because the inner function may outlive the call that created them. The cell is allocated when the outer function runs, and every inner function that mentions the name points at that same cell. `cell_contents` reads it. Assignment in the outer function, or a `nonlocal` assignment in the inner one, writes it. There is one box per closed-over name, not one box per inner function.

That single box is why the loop surprises you. `for number in range(3)` assigns the same local `number` three times. The three `report` functions all point at that one cell. Calling them later reads the current contents — the last assigned value. Nothing in the `def` copied `number`. A default argument does copy, because the default expression runs at definition time and the result is stored on the function object in `__defaults__`.

The compiler decides local versus enclosing before the function runs. If the body assigns to a name, that name is local for the whole body. A read before the assignment loads a local that has not been written yet — `UnboundLocalError` rather than falling through to the global. Delete the assignment and the same read would find a global.

`global` and `nonlocal` opt into a different slot. `global` names a module attribute. `nonlocal` names a cell in an enclosing function. They do not create a variable in an outer scope that does not already have the name.

A class body looks like a scope and is not an enclosing function. While the class statement runs, names go into a temporary dict that becomes the class. A comprehension is a nested function and does not look in class bodies — `[x for i in range(1)]` inside the class raises `NameError` for `x` even when `x = 2` is a line above. A method reads `self.x` by attribute lookup, a different mechanism.

The design reason for cells instead of snapshots is that nested functions may share mutable state on purpose. The cost is the late read — the loop bug. Defaults, or a factory that takes the value as a parameter, are how you ask for a snapshot.


In [3]:
def counter():
    count = 0

    def bump():
        nonlocal count
        count += 1
        return count

    return bump


bump = counter()
print(bump.__closure__[0].cell_contents)
print(bump(), bump())
print(bump.__closure__[0].cell_contents)

0
1 2
2


In [4]:
class Room:
    code = "a"
    labels = [code for _ in range(1)]

NameError: name 'code' is not defined

## When to use it

Close over a name when several inner functions should share one piece of state, and use `nonlocal` if an inner function must assign it. Snapshot with a default, or with a small factory that takes the value as a parameter, when each function needs the value from one iteration.

Use `global` only to rebind a module-level name on purpose. Reading a global needs no declaration. Assigning without `global` creates a local and hides the module name.

Do not expect a comprehension in a class body to see the class's names. Compute the list outside the class, or in a method that receives `self`.


## Common mistake

Lambdas created in a loop share the loop variable. Called after the loop, they all return the last value. The functions are distinct objects. The cell is not. Nothing raises, and a test that only checks the length of the list still passes.


In [5]:
def make_late():
    reporters = []
    for number in range(3):
        reporters.append(lambda: number)
    return reporters


print([report() for report in make_late()])

[2, 2, 2]


The print is `[2, 2, 2]`. Each lambda reads `number` when it is called, and the loop has finished. The fix stores the current integer as a default, which is evaluated at `def` time. The lambdas then return 0, 1, and 2.


In [6]:
def make_fixed():
    reporters = []
    for number in range(3):
        reporters.append(lambda number=number: number)
    return reporters


print([report() for report in make_fixed()])

[0, 1, 2]


Reading a local before its assignment raises `UnboundLocalError`. The last line of the traceback names that error. The name is local because of the assignment below the read, not because a global is missing.


In [7]:
def surprise():
    print(n)
    n = 1


surprise()

UnboundLocalError: cannot access local variable 'n' where it is not associated with a value

## Try it

Change `tariffs` so the inner function does not use a default, and instead calls a tiny factory `def at(rate):` that returns the biller. Why does that also snapshot the rate, and which name does the inner function look up?


## What this explains

- [Scope](../../course/05-functions/03-scope-lesson.ipynb): a name assigned in a function is local. LEGB is the lookup order behind that rule.
- [Late binding](../../course/10-pitfalls/04-late-binding-lesson.ipynb): the loop closure reads the cell at call time. The default-argument fix stores a separate value.
- [Numbers and strings](../05-numbers/01-numbers-strings-lesson.ipynb): the cell holds a pointer to an object. Rebinding the name changes the pointer in the cell. It does not edit the integer.


## Recap

- Lookup is local, then enclosing cells, then global, then builtins. An assignment makes a name local for the whole function.
- A closure shares a cell. The value is read at call time. A default argument snapshots it at `def` time.
- A class body is not an enclosing scope. A comprehension there cannot see the class's names.

Next: [The iteration protocol](../07-iteration/01-protocol-lesson.ipynb).
